In [ ]:
import onnx


# Funzione per ottenere i parametri del modello ONNX
def get_model_parameters(model_path):
    model = onnx.load(model_path)

    params = {}
    for initializer in model.graph.initializer:
        weight_or_bias = onnx.numpy_helper.to_array(initializer)
        params[initializer.name] = weight_or_bias
        print(f"Name: {initializer.name}")
        print(f"Shape: {weight_or_bias.shape}")
        #print(f"Data:\n{weight_or_bias}\n")

    return params



In [ ]:
get_model_parameters(r"C:\Users\andr3\PycharmProjects\pyNeVer\examples\pruning_experiments\results\no_batch\90.onnx")

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define the neural network class
class CustomNN(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, filters_number: int, kernel_size: int, stride: int,
                 padding: int, hidden_layer_dim: int):
        super(CustomNN, self).__init__()

        # Convolutional layer
        self.conv = nn.Conv2d(1, filters_number, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn1 = nn.BatchNorm2d(filters_number)  # Batch normalization for conv layer
        self.flatten = nn.Flatten()

        # Calculate input size for fc1 dynamically
        dummy_input = torch.zeros(1, 1, input_dim, input_dim)
        conv_output = self.conv(dummy_input)
        conv_output = self.bn1(conv_output)  # Ensure BatchNorm compatibility
        conv_output_flatten = self.flatten(conv_output)
        fc1_in_features = conv_output_flatten.numel()

        # Fully connected layers
        self.fc1 = nn.Linear(fc1_in_features, hidden_layer_dim)
        self.fc1_dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(hidden_layer_dim, output_dim)

    def forward(self, x):
        # Apply convolution, batch norm, and ReLU
        x = self.conv(x)
        x = self.bn1(x)
        x = F.relu(x)

        # Fully connected layers
        x = self.flatten(x)
        x = self.fc1_dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# Hyperparameters
input_dim = 28  # Assuming MNIST dataset
output_dim = 10
filters_number = 25
kernel_size = 3
stride = 1
padding = 0
hidden_layer_dim = 48
batch_size = 128
epochs = 100
learning_rate = 0.001

# Data loading and preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model, loss function, and optimizer
model = CustomNN(input_dim, output_dim, filters_number, kernel_size, stride, padding, hidden_layer_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        # Move data to device if using GPU
        images, labels = images, labels

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {total_loss:.4f}, Accuracy: {train_accuracy:.2f}%")

# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")


Epoch [1/100], Loss: 303.5242, Accuracy: 77.93%
Epoch [2/100], Loss: 170.1499, Accuracy: 87.37%
Epoch [3/100], Loss: 146.1532, Accuracy: 89.05%
Epoch [4/100], Loss: 132.5185, Accuracy: 89.97%
Epoch [5/100], Loss: 121.6446, Accuracy: 90.81%
Epoch [6/100], Loss: 113.9171, Accuracy: 91.30%
Epoch [7/100], Loss: 107.5440, Accuracy: 91.71%
Epoch [8/100], Loss: 99.1485, Accuracy: 92.30%
Epoch [9/100], Loss: 92.6081, Accuracy: 92.81%
Epoch [10/100], Loss: 87.4789, Accuracy: 93.27%
Epoch [11/100], Loss: 81.9271, Accuracy: 93.72%
Epoch [12/100], Loss: 78.4903, Accuracy: 93.88%
Epoch [13/100], Loss: 77.0132, Accuracy: 94.09%
Epoch [14/100], Loss: 71.5613, Accuracy: 94.50%
Epoch [15/100], Loss: 68.7233, Accuracy: 94.73%
Epoch [16/100], Loss: 65.9279, Accuracy: 94.89%
Epoch [17/100], Loss: 62.4115, Accuracy: 95.14%
Epoch [18/100], Loss: 60.6988, Accuracy: 95.33%
Epoch [19/100], Loss: 59.5763, Accuracy: 95.34%
Epoch [20/100], Loss: 57.5550, Accuracy: 95.50%
Epoch [21/100], Loss: 55.7503, Accuracy: 9